# Total return swaps and variance swaps

**Start here:** This deep dive expands on `02_pricing/pricing_across_asset_classes.ipynb`; use `02_pricing/pricing_fundamentals.ipynb` for the shared instrument JSON -> market -> model pipeline.

**Prerequisites:** `02_pricing/pricing_fundamentals.ipynb` and `02_pricing/pricing_across_asset_classes.ipynb`. **`trs_equity`** for index TRS vs a **financing leg**; **`variance_swap`** for realized vs strike variance with **`vega`-style** metrics when registered.


## Concept

- **TRS:** receive/pay **total return** (price + dividends) vs **funded financing** (curve + spread).
- **Variance swap:** payoff on realized variance vs **strike variance**; **vega notional** scales sensitivity.

Variance swap PV may be near zero without a full forward variance / surface stack—focus on the metric plumbing.


In [ ]:
import json
from datetime import date

from finstack_quant.valuations import ValuationResult
from finstack_quant.valuations.instruments import price_instrument_with_metrics, validate_instrument_json
from finstack_quant.core.market_data import DiscountCurve, ForwardCurve, MarketContext

print("Imports OK (TRS / variance).")


## Instrument JSON


In [ ]:
AS_OF = date(2025, 1, 15)
AS_OF_STR = AS_OF.isoformat()

trs = json.dumps(
    {
        "type": "trs_equity",
        "spec": {
            "id": "TRS-SPX-1Y",
            "side": "receive_total_return",
            "notional": {"amount": "5000000", "currency": "USD"},
            "schedule": {
                "start": AS_OF_STR,
                "end": "2026-01-15",
                "params": {
                    "freq": {"count": 3, "unit": "months"},
                    "dc": "Act360",
                    "bdc": "following",
                    "calendar_id": "weekends_only",
                    "end_of_month": False,
                    "payment_lag_days": 0,
                    "stub": "None",
                },
            },
            "underlying": {
                "ticker": "SPX",
                "currency": "USD",
                "contract_size": 1.0,
                "spot_id": "SPX-SPOT",
                "div_yield_id": "SPX-DIV",
            },
            "financing": {
                "discount_curve_id": "USD-OIS",
                "forward_curve_id": "USD-SOFR-3M",
                "spread_bp": "75",
                "day_count": "Act360",
            },
            "dividend_tax_rate": 0.0,
            "attributes": {},
            "pricing_overrides": {},
        },
    }
)

var_swap = json.dumps(
    {
        "type": "variance_swap",
        "spec": {
            "id": "VARSPX-1Y",
            "side": "receive",
            "notional": {"amount": "1000000", "currency": "USD"},
            "start_date": AS_OF_STR,
            "maturity": "2026-01-15",
            "strike_variance": 0.04,
            "underlying_ticker": "SPX",
            "observation_freq": {"count": 1, "unit": "days"},
            "observation_calendar_id": "weekends_only",
            "realized_var_method": "CloseToClose",
            "day_count": "Act365F",
            "discount_curve_id": "USD-OIS",
            "attributes": {},
            "pricing_overrides": {},
        },
    }
)

for label, js in (("trs_equity", trs), ("variance_swap", var_swap)):
    validate_instrument_json(js)
    print(label, "JSON OK")


## MarketContext


In [ ]:
ois = DiscountCurve(
    "USD-OIS",
    AS_OF,
    [(0.0, 1.0), (1.0, 0.97), (5.0, 0.85)],
    day_count="act_365f",
)
sofr = ForwardCurve(
    "USD-SOFR-3M",
    0.25,
    knots=[(0.0, 0.05), (5.0, 0.045)],
    base_date=AS_OF,
    day_count="act_360",
)
mc = MarketContext().insert(ois).insert(sofr)
mc.insert_price("SPX-SPOT", 4500.0, "USD")
mc.insert_price("SPX-DIV", 0.015)
mc.insert_price("SPX_IMPL_VOL", 0.20)
market_json = mc.to_json()
print("market ready for TRS + variance")

## Pricing


In [ ]:
for label, js in (("trs", trs), ("variance", var_swap)):
    out = price_instrument_with_metrics(js, market_json, AS_OF_STR, model="discounting")
    print(label, ValuationResult.from_json(out))


## Metrics


In [ ]:
for label, js, mets in (
    ("trs", trs, ["financing_annuity", "index_delta", "equity_value"]),
    ("variance", var_swap, ["vega", "implied_vol", "variance_vega"]),
):
    out = price_instrument_with_metrics(
        js, market_json, AS_OF_STR, model="discounting", metrics=mets
    )
    vr = ValuationResult.from_json(out)
    print("—", label)
    for m in mets:
        v = vr.get_metric(m)
        if v is not None:
            print(f"  {m}: {v}")
print("Vega notional scales variance swap sensitivity to implied vol changes (see deal terms).")


## Takeaways

- **TRS** splits **asset leg** vs **financing leg**; curves must include `forward_curve_id` for float financing.
- **Variance swaps** need rich forward variance inputs for non-trivial PV; metrics still demonstrate the hook.
- Always **`validate_instrument_json`** after edits to long JSON specs.
